# Notebook 13 — Hyper-Optimization Sprints (Break 0.80 F1)

Foundation: **Frozen Golden Baseline** (`unitary/toxic-bert`, 6-label sigmoid `toxic` score).

**Objective:** Test F1 weighted **> 0.80** with train–test gap **< 5%** (briefing rule).

| Exp | Method | 5-Fold CV |
|-----|--------|----------|
| **1** | Multi-pivot aug (DE/FR/ES) + head-only train | ✅ |
| **2** | Advanced TTA (Original + DE + FR weighted) | ✅ |
| **3** | CLS hidden states + style meta → LR C=0.01 | ✅ |
| **4** | Ultra-fine threshold (0.05–0.30, step 0.001) on best of 1–3 | ✅ |

Artifacts: `models/notebook_13/` · Reports: `reports/notebook_13/sprint_results.json`

```bash
uv run python -m src.experiments.notebook_13_sprints
```

## 0. Setup

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists() and (PROJECT_ROOT.parent / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACT_DIR = PROJECT_ROOT / "models" / "notebook_13"
REPORT_DIR = PROJECT_ROOT / "reports" / "notebook_13"
RESULTS_PATH = REPORT_DIR / "sprint_results.json"
print(ARTIFACT_DIR)
print(RESULTS_PATH)

## 1. Run all sprints (long-running — translation + CV)

In [ ]:
from src.experiments.notebook_13_sprints import main

main()

## 2. Load results (if already executed)

In [ ]:
if not RESULTS_PATH.exists():
    raise FileNotFoundError(f"Run sprints first: uv run python -m src.experiments.notebook_13_sprints")

results = json.loads(RESULTS_PATH.read_text())
comparison = pd.DataFrame(results["comparison_table"])
comparison

## 3. Per-fold gap monitor

In [ ]:
rows = []
for key in ("golden_baseline_cv", "exp1", "exp2", "exp3", "exp4"):
    block = results.get(key, {})
    for f in block.get("folds", []):
        rows.append({
            "experiment": key,
            "fold": f["fold"],
            "f1_test": f["f1_test"],
            "gap_pp": f["train_test_gap_pp"],
            "gap_ok": f["gap_ok"],
            "status": "PASS" if f["gap_ok"] else "FAIL_GAP",
        })
pd.DataFrame(rows).pivot_table(
    index="experiment", values=["f1_test", "gap_pp"], aggfunc=["mean", "max"]
)

## 4. Comparison markdown report

In [ ]:
from IPython.display import Markdown, display

md = REPORT_DIR / "comparison_table.md"
if md.exists():
    display(Markdown(md.read_text()))

## Conclusion

**Sprint results:** `reports/notebook_13/sprint_results.json`

| Sprint | Mean F1 (test) | Max gap (pp) | All folds gap OK | Mean F1 ≥ 0.80 |
|--------|----------------|--------------|------------------|----------------|
| Golden Baseline (CV) | 0.7748 | 8.09 | ❌ | ❌ |
| Exp1 Multi-Pivot + Head | 0.7493 | 12.42 | ❌ | ❌ |
| Exp2 Advanced TTA | 0.7592 | 6.53 | ❌ | ❌ |
| Exp3 Meta Stacking | **0.7894** | 9.77 | ❌ | ❌ |
| Exp4 Ultra-Fine Thresh | 0.7704 | 9.42 | ❌ | ❌ |

**Which sprint reached 0.80?** No sprint passed **both** constraints on all 5 folds. Best single folds: **Exp3 fold 0** (F1=0.8147, gap=3.39 pp) and **Exp4 fold 4** (F1=0.8083, gap=0.18 pp, threshold≈0.299).

**Final train–test gap:** Best average gap discipline: Golden Baseline / Exp2 TTA (~3.3–3.6 pp mean). Exp3 has highest mean F1 but **FAIL_GAP** (6.94 pp mean).

**Production recommendation:** **Frozen Golden Baseline** for briefing compliance (~0.77–0.79 CV F1, minimal overfit). Exp3+Exp4 threshold tuning is promising on individual folds but not stable across CV.

Artifacts: `models/notebook_13/` (augment cache, head-only checkpoints).